In [1]:
import os
from pathlib import Path
from openai import OpenAI
from langchain_openai import ChatOpenAI

from ontovis.vis_agent import *

In [2]:
llm = ChatOpenAI(
    model="gpt-5.4",
    api_key=os.environ["OPENAI_API_KEY"]
)

In [3]:
filename = "mri_woman_256x256x109_uint16"
description = "MRI scan of a woman's head"

workspace_prefix = "mri_woman_head"
folder = "/home/pascalgrosset/projects/OntoVis/resources/datasets/"
fullpath = folder + filename

In [4]:
vis_ai = VisExplorer(llm, config_path="../configs/config.yaml", workspace_prefix=workspace_prefix)

Creating workspace at: /home/pascalgrosset/projects/OntoVis/notebooks/mri_woman_head_vis_agent__2026_04_06__17_16


In [5]:
vis_ai.chat(f"Can you use the vtk package and volume render the at: {fullpath}, which is a {description}. Render images that are suitable for medical professionals. Produce 20 different images and create a file listing, called manifest.txt. that list the images created and their absolute path")


-----volume_rendering_instructions---



Python REPL can execute arbitrary code. Use with caution.



-----python_repl_tool---


-----python_repl_tool---


-----python_repl_tool---


-----python_repl_tool---


-----python_repl_tool---


-----python_repl_tool---



Done — I used VTK to volume render the MRI head dataset and created 20 PNG images plus a `manifest.txt` with filenames and absolute paths.

Files are in:
`/home/pascalgrosset/projects/OntoVis/notebooks/mri_woman_head_vis_agent__2026_04_06__17_16`

Manifest:
`/home/pascalgrosset/projects/OntoVis/notebooks/mri_woman_head_vis_agent__2026_04_06__17_16/manifest.txt`

Also created:
- `histogram_mri_woman.png`
- `render_mri.py`

The 20 rendered images are:
1. `01_ap_soft.png`
2. `02_pa_soft.png`
3. `03_left_lat_soft.png`
4. `04_right_lat_soft.png`
5. `05_superior_soft.png`
6. `06_inferior_soft.png`
7. `07_ap_cortical.png`
8. `08_left_lat_cortical.png`
9. `09_right_oblique_30.png`
10. `10_left_oblique_30.png`
11. `11_ap_deep.png`
12. `12_left_lat_deep.png`
13. `13_right_lat_deep.png`
14. `14_superior_deep.png`
15. `15_pa_outline.png`
16. `16_right_oblique_outline.png`
17. `17_left_oblique_outline.png`
18. `18_cranial_oblique.png`
19. `19_caudal_oblique.png`
20. `20_three_quarter_soft.png`

Note: the dataset path you gave appears to refer to the raw volume without the `.raw` extension in the actual filesystem, so I used:
`/home/pascalgrosset/projects/OntoVis/resources/datasets/mri_woman_256x256x109_uint16.raw`

If you want, I can also generate a second set with more diagnostic transfer functions focused on:
- skin/scalp suppression
- white/gray matter contrast
- vessel-like bright-structure emphasis


Query took: 171.34 seconds, total tokens used: 18306
 


In [ ]:
vis_ai.chat("Can you circle around the aneurism (not over) in the images using the manifest at /home/pascalgrosset/projects/OntoVis/notebooks/vis_agent__2026_04_06__10_24/manifest.txt and explain your reasoning")

In [ ]:
rubric = """
domain: "CT 3D volume rendering (bone-focused)"
goal: >
  Rate images (0–10) for reference-grade CT 3D rendering aesthetics:
  clean background, disciplined opacity/transfer function, sharp edges,
  minimal artifacts, balanced lighting, and clinically standard viewpoints.

inputs:
  - image: "single CT 3D render (PNG/JPG), typically skull/bone"

outputs:
  fields:
    - final_score: "float, 0.0–10.0"
    - cap_applied: "null or one of [G1,G2,G3,G4,G5]"
    - component_scores:
        BG: "0–5"
        OP: "0–5"
        ED: "0–5"
        AR: "0–5"
        LT: "0–5"
        VW: "0–5"
    - top_penalties: "list of up to 2 reason codes from [BG,OP,ED,AR,LT,VW,AN]"
    - one_line_rationale: "single sentence"
    - improvement_hint: "single sentence"

reason_codes:
  BG: "background/framing hygiene issues"
  OP: "opacity/transfer function haze or poor bone isolation"
  ED: "edge/detail softness or aliasing"
  AR: "artifacts: banding/striations/ripple/floating fragments"
  LT: "lighting issues: clipping/hotspots or underexposure"
  VW: "viewpoint/composition not clinically standard"
  AN: "annotation/watermark clutter"

procedure:
  step_1_gate_caps:
    description: >
      Check for failure modes; if present, cap the final score regardless of component sum.
    gates:
      - id: G1
        condition: "major anatomy clipped/truncated (skull not fully in frame)"
        cap: 7.5
      - id: G2
        condition: "strong corner/edge stray geometry or obvious non-anatomical fragments"
        cap: 8.0
      - id: G3
        condition: "heavy opacity fog/smoke obscures key landmarks"
        cap: 7.0
      - id: G4
        condition: "severe highlight clipping (large blown-out regions)"
        cap: 8.0
      - id: G5
        condition: "severe banding/striations/ripple dominating surfaces"
        cap: 7.5
    rule: >
      If multiple gates trigger, use the lowest cap (most restrictive).
      If none trigger, cap_applied = null and cap = 10.0.

  step_2_component_scoring:
    scale: "Each component ri is rated 0–5 using anchors below."
    weights:
      BG: 0.18
      OP: 0.24
      ED: 0.20
      AR: 0.18
      LT: 0.12
      VW: 0.08
    formula:
      base_score: "10 * ( Σ_i ( w_i * (r_i / 5) ) )"
      final_score: "min(cap, base_score)"
    anchors:
      BG:
        5: "uniform black background; skull fully in frame; clean silhouette; no edge clutter"
        3: "minor vignette/edge distractions; slight framing imperfections"
        1: "noticeable cropping risk; messy borders; distracting peripheral elements"
        0: "clear truncation/clipping; dominant border artifacts"
      OP:
        5: "bone appears surface-like; minimal low-density haze; stable tone across skull"
        3: "some fog/haze but landmarks remain readable"
        1: "significant smoke veil in midface/cranial vault; poor separation"
        0: "opacity mapping obscures anatomy; rendering looks cloudy/composited"
      ED:
        5: "crisp teeth/orbits/nasal aperture; no obvious blur; minimal aliasing"
        3: "mild softness or mild jagged edges"
        1: "blur/smear; fine bony boundaries poorly defined"
        0: "detail largely lost; edges unreliable"
      AR:
        5: "minimal banding/striations/ripple; no floating fragments"
        3: "mild artifacts visible but not attention-grabbing"
        1: "artifacts compete with anatomy (banding/ripple/floats)"
        0: "dominant artifacts; anatomy hard to read"
      LT:
        5: "controlled highlights; shadows add depth without hiding anatomy"
        3: "slightly hot teeth/forehead or slightly dark orbits"
        1: "large hotspots or underexposed regions hide structure"
        0: "lighting undermines readability (extreme clipping or darkness)"
      VW:
        5: "textbook frontal or 3/4; symmetry/landmarks optimized"
        3: "acceptable angle but not optimal"
        1: "awkward view obscures key anatomy"
        0: "viewpoint prevents clinical interpretation"

  step_3_penalties_and_text:
    top_penalties_rule: >
      Choose up to 2 reason codes corresponding to the largest visible deficiencies
      (or the lowest component scores). If an annotation/watermark is prominent,
      include AN as a penalty.
    one_line_rationale_template: >
      "Score reflects {strengths}; deductions mainly for {top_penalties}."
    improvement_hint_template: >
      "To improve: {single highest-impact adjustment aligned with penalties}."

calibration:
  reference_grade_threshold:
    rule: >
      Only allow scores >= 9.5 if:
      OP >= 4.5 AND ED >= 4.5 AND BG >= 4.5 AND AR >= 4.0 AND LT >= 4.0
      AND no gate caps triggered.
  note: >
    This rubric is tuned to "reference CT 3D" aesthetics rather than artistic style.

example_output:
  final_score: 8.6
  cap_applied: null
  component_scores: {BG: 4, OP: 4, ED: 4, AR: 4, LT: 3, VW: 5}
  top_penalties: ["LT", "OP"]
  one_line_rationale: "Strong framing and viewpoint with good bone emphasis; deductions mainly for LT and OP."
  improvement_hint: "Reduce hotspot intensity and slightly tighten the opacity window to suppress residual haze."
"""

In [ ]:
vis_ai.chat(f"using this {rubric} reevalute the images at created_images.txt")

In [ ]:
ref_files = [
    "/home/pascalgrosset/projects/OntoVis/knowledge/ref_imgs/Screenshot 2026-02-25 095520.png",
    "/home/pascalgrosset/projects/OntoVis/knowledge/ref_imgs/Screenshot 2026-02-25 095614.png",
    "/home/pascalgrosset/projects/OntoVis/knowledge/ref_imgs/Screenshot 2026-02-25 095646.png",
    "/home/pascalgrosset/projects/OntoVis/knowledge/ref_imgs/Screenshot 2026-02-25 095734.png",
    "/home/pascalgrosset/projects/OntoVis/knowledge/ref_imgs/Screenshot 2026-02-25 095922.png",
]

In [ ]:
vis_ai.chat(f"the images at {ref_files} show the desired resdering for that dataset. Can you evaluate the image and see how they are good and then re-render 5 suitable images from the file at /home/pascalgrosset/projects/OntoVis/notebooks/skull_256x256x256_uint8.raw")

In [ ]:
vis_ai.chat("can you regenerate that")